# HealthifyAI Backend — Colab + Flask + ngrok
**Before running:** Runtime → Change runtime type → T4 GPU

In [1]:
!pip install -q transformers==4.44.2 accelerate==0.33.0 bitsandbytes==0.45.5
!pip install -q pyngrok flask flask-cors pillow pytesseract
!apt-get install -q tesseract-ocr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 78.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires num

In [1]:
# Cell 2 — Authenticate ngrok
from pyngrok import ngrok

# Get your token from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = # <-- paste your token here
ngrok.set_auth_token(NGROK_TOKEN)

In [2]:
# Cell 3 — Load model in 4-bit (fits in T4's 15GB VRAM)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "adarsh6/healthifyai-base-lora-merged"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading tokenizer...")
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model in 4-bit... (takes ~2-3 min)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    # device_map={"": 0},
    trust_remote_code=True
)
model.eval()
print("Model loaded!")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:174: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded!


In [3]:
# Cell 4 — OCR helper
import pytesseract
from PIL import Image
import re

def extract_text_from_image(image: Image.Image) -> str:
    text = pytesseract.image_to_string(image)
    # Basic cleaning
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def run_inference(extracted_text: str) -> str:
    prompt = (
        f"[INST] You are a health expert. Analyze the following food product ingredients "
        f"and provide a detailed health review including potential risks, benefits, and suitability "
        f"for common health conditions.\n\nIngredients: {extracted_text} [/INST]"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    # Return only the generated part, not the prompt
    return response[len(prompt):].strip()

In [4]:
# Cell 5 — Flask app
import base64
import io
import threading
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)  # Allow requests from your Vercel frontend

@app.route("/", methods=["GET"])
def health_check():
    return jsonify({"status": "ok", "message": "HealthifyAI backend is running"})

@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.json

        # Accepts either raw text OR base64 image
        if "text" in data:
            extracted_text = data["text"]

        elif "image" in data:
            # base64 encoded image from frontend
            image_data = base64.b64decode(data["image"])
            image = Image.open(io.BytesIO(image_data))
            extracted_text = extract_text_from_image(image)

            if not extracted_text:
                return jsonify({"error": "Could not extract text from image"}), 400
        else:
            return jsonify({"error": "Provide either 'text' or 'image' in request body"}), 400

        result = run_inference(extracted_text)
        return jsonify({
            "extracted_text": extracted_text,
            "review": result
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 500

# Run Flask in a background thread so the notebook doesn't block
thread = threading.Thread(target=lambda: app.run(port=5000, use_reloader=False))
thread.daemon = True
thread.start()
print("Flask server started on port 5000")

Flask server started on port 5000


In [5]:
# Cell 6 — Start ngrok tunnel with your static domain
from pyngrok import ngrok

STATIC_DOMAIN = "unifier-sleek-cornhusk.ngrok-free.dev"

tunnel = ngrok.connect(5000, domain=STATIC_DOMAIN)
print(f"\n Backend live at: https://{STATIC_DOMAIN}")
print(f" Predict endpoint: https://{STATIC_DOMAIN}/predict")
print("\n Set this as your API URL in Vercel frontend env vars.")


 Backend live at: https://unifier-sleek-cornhusk.ngrok-free.dev
 Predict endpoint: https://unifier-sleek-cornhusk.ngrok-free.dev/predict

 Set this as your API URL in Vercel frontend env vars.


In [ ]:

import time

print("Keeping session alive... (Ctrl+C or interrupt kernel to stop)")
while True:
    time.sleep(600)
    print("still running...")